In [33]:
import pymongo
from pymongo import MongoClient


In [34]:
#Celda 1 — Conexión y ping

# Dirección del servidor MongoDB local.
# "localhost" = tu propio ordenador, 27017 = puerto por defecto de MongoDB.
uri = "mongodb://localhost:27017"

# Creamos el cliente: el objeto que representa la conexión con el servidor.
# Ojo: crear el cliente todavía NO comprueba que el servidor responda.
cliente = MongoClient(uri)

# Accedemos a la base de datos de sistema "admin" (siempre existe)
# y le enviamos el comando "ping" para obligar al servidor a contestar.
respuesta = cliente["admin"].command("ping")

# Si todo ha ido bien, la respuesta es {'ok': 1.0}
print(respuesta)

{'ok': 1.0}


In [35]:
#Celda 2 — Paso 5: crear la base de datos y la colección

# Mostramos las bases de datos que existen ANTES de crear la nuestra.
# Solo deberían aparecer las de sistema: admin, config y local.
print("Bases de datos antes:", cliente.list_database_names())

# Accedemos a la base de datos "RecursosHumanos".
# Esto solo crea la referencia en Python; en el servidor aún no existe.
db = cliente["RecursosHumanos"]

# Si la colección ya existe (porque ejecutamos el notebook otra vez),
# la borramos para empezar siempre desde cero con los datos originales.
# Si no existe, esta línea no hace nada y no da error.
db.drop_collection("Curriculums")

# Creamos la colección de forma explícita.
# En este momento SÍ se crean en el servidor la colección y la base de datos.
db.create_collection("Curriculums")

# Guardamos la colección en una variable para usarla en el resto del notebook
coleccion = db["Curriculums"]

# Comprobamos que ahora sí existen
print("Bases de datos después:", cliente.list_database_names())
print("Colecciones en RecursosHumanos:", db.list_collection_names())        

    

Bases de datos antes: ['RecursosHumanos', 'admin', 'config', 'local']
Bases de datos después: ['RecursosHumanos', 'admin', 'config', 'local']
Colecciones en RecursosHumanos: ['Curriculums']


In [36]:
#Celda 3 — Paso 6: cargar el JSON e insertar los documentos

# Importamos el módulo json de Python para leer el archivo.
# (Lo ideal es mover este import a la celda 0 junto con el de pymongo.)
import json

# Ruta al archivo. El notebook está en src/ y el JSON en data/,
# así que subimos un nivel con ".." y entramos en data/.
ruta = "../data/curriculumns.json"

# Abrimos el archivo en modo lectura con codificación UTF-8
# y convertimos su contenido en objetos de Python con json.load().
with open(ruta, encoding="utf-8") as archivo:
    curriculums = json.load(archivo)

# El JSON es una lista de diccionarios: cada diccionario es un currículum
print("Documentos leídos del archivo:", len(curriculums))

# Insertamos todos los documentos de golpe con insert_many,
# porque tenemos una lista (insert_one es para un solo documento).
resultado = coleccion.insert_many(curriculums)

# insert_many devuelve los _id que MongoDB ha generado para cada documento
print("Documentos insertados:", len(resultado.inserted_ids))
print("IDs generados:", resultado.inserted_ids)

Documentos leídos del archivo: 3
Documentos insertados: 3
IDs generados: [ObjectId('6ac77f16dfe13829d8af7378'), ObjectId('6ac77f16dfe13829d8af7379'), ObjectId('6ac77f16dfe13829d8af737a')]


In [37]:
#Celda 4 — Paso 7: consultar e imprimir todos los documentos

# Importamos pprint ("pretty print"), que imprime diccionarios largos
# con sangría y saltos de línea, para que se lean mucho mejor que con print.
# (Lo ideal es mover este import a la celda 0 con los demás.)
from pprint import pprint

# find() busca documentos en la colección.
# Le pasamos un filtro vacío {} → significa "sin condiciones", es decir, TODOS.
# find() no devuelve una lista, sino un CURSOR: un objeto que va
# entregando los documentos de uno en uno cuando lo recorremos.
documentos = coleccion.find({})

# Recorremos el cursor con un for e imprimimos cada documento

for doc in documentos:
    
    pprint(doc)
    print("-" * 60)   # línea separadora entre documentos

{'_id': ObjectId('6ac77f16dfe13829d8af7378'),
 'contacto': {'blog': 'thegatesnotes.com', 'twitter': '@BillGates'},
 'experiencia': [{'empresa': 'Bill & Melinda Gates Foundation',
                  'fin': 'Presente',
                  'inicio': 2000,
                  'puesto': 'Co-chair'},
                 {'empresa': 'Microsoft',
                  'fin': 'Presente',
                  'inicio': 1975,
                  'puesto': 'Co-founder, Chairman'}],
 'formacion': [{'fin': 1975,
                'inicio': 1973,
                'institucion': 'Harvard University'},
               {'fin': 1973,
                'inicio': 1967,
                'institucion': 'Lakeside School, Seattle'}],
 'nombre': 'Bill Gates',
 'resumen': 'Co-chair of the Bill & Melinda Gates Foundation. Chairman, '
            'Microsoft Corporation.',
 'titular': 'Greater Seattle Area | Philanthropy'}
------------------------------------------------------------
{'_id': ObjectId('6ac77f16dfe13829d8af7379'),
 'contacto

In [38]:
#Celda extra

#contar cuantos documentos tengo
total = coleccion.count_documents({})
print("Documentos en Curriculums:", total)

Documentos en Curriculums: 3


In [39]:
#Celda 5 — Paso 8: encontrar a "Bill Gates" e imprimir su nombre y titular

# find_one() devuelve UN solo documento (el primero que cumpla el filtro),
# directamente como diccionario, sin cursor. Si no encuentra nada devuelve None.
#
# Primer argumento → FILTRO: qué documento buscamos (nombre igual a "Bill Gates").
# Segundo argumento → PROYECCIÓN: qué campos queremos que nos devuelva.
#   1 = incluir el campo, 0 = excluirlo.
#   _id siempre se devuelve salvo que lo excluyamos explícitamente con 0.
bill = coleccion.find_one(
    {"nombre": "Bill Gates"},
    {"nombre": 1, "titular": 1, "_id": 0}
)

# Comprobamos que lo ha encontrado antes de imprimir
if bill:
    print("Nombre: ", bill["nombre"])
    print("Titular:", bill["titular"])
else:
    print("No se ha encontrado a Bill Gates")

Nombre:  Bill Gates
Titular: Greater Seattle Area | Philanthropy


In [40]:
#Celda 6 — Paso 9: nombre y titular de quienes tienen experiencia en "Linux Foundation"

# "experiencia" es una LISTA de diccionarios, y "empresa" está dentro de cada uno.
# Con la notación con punto "experiencia.empresa" MongoDB busca dentro de
# cada elemento de la lista: basta con que UNO de ellos coincida.
filtro = {"experiencia.empresa": "Linux Foundation"}

# Proyección: solo nombre y titular, sin el _id
proyeccion = {"nombre": 1, "titular": 1, "_id": 0}

# find() porque podría haber varias personas que cumplan la condición
for persona in coleccion.find(filtro, proyeccion):
    print("Nombre: ", persona["nombre"])
    print("Titular:", persona["titular"])
    print("-" * 60)

Nombre:  Linus Torvalds
Titular: Portland Area | Open Source Software
------------------------------------------------------------


In [41]:
#Celda 7 — Paso 10: personas que empezaron su formación en 1973
# Imprimir nombre, institución, fecha de inicio y de fin

# Filtro con notación con punto sobre la lista "formacion"
filtro = {"formacion.inicio": 1973}

# Proyección con el OPERADOR POSICIONAL "$":
# "formacion.$" significa "devuélveme solo el elemento de la lista formacion
# que ha hecho que se cumpla el filtro". Sin él saldrían TODAS las formaciones
# de la persona (Bill Gates tiene dos y solo una empezó en 1973).
proyeccion = {"nombre": 1, "formacion.$": 1, "_id": 0}

for persona in coleccion.find(filtro, proyeccion):
    # Aunque solo venga un elemento, "formacion" sigue siendo una lista
    for estudio in persona["formacion"]:
        print("Nombre:     ", persona["nombre"])
        print("Institución:", estudio["institucion"])
        print("Inicio:     ", estudio["inicio"])
        print("Fin:        ", estudio["fin"])
        print("-" * 60)

Nombre:      Bill Gates
Institución: Harvard University
Inicio:      1973
Fin:         1975
------------------------------------------------------------


In [42]:
#Celda 8 — Paso 11: formación cursada entre 1980 y 1990
# Imprimir nombre e instituciones

# Interpretación: la formación EMPEZÓ y TERMINÓ dentro del rango 1980-1990.
inicio_min = 1980
fin_max = 1990

# $gte = "mayor o igual que" (>=)   $lte = "menor o igual que" (<=)
#
# Usamos $elemMatch para que las DOS condiciones se cumplan en el MISMO
# elemento de la lista "formacion". Si las pusiéramos sueltas
# ({"formacion.inicio": {...}, "formacion.fin": {...}}), MongoDB podría
# cumplir una con un estudio y la otra con un estudio distinto.
filtro = {
    "formacion": {
        "$elemMatch": {
            "inicio": {"$gte": inicio_min},
            "fin": {"$lte": fin_max}
        }
    }
}
proyeccion = {"nombre": 1, "formacion": 1, "_id": 0}

for persona in coleccion.find(filtro, proyeccion):
    # MongoDB nos devuelve la persona con TODAS sus formaciones,
    # así que en Python nos quedamos solo con las que están dentro del rango.
    instituciones = [
        estudio["institucion"]
        for estudio in persona["formacion"]
        if estudio["inicio"] >= inicio_min and estudio["fin"] <= fin_max
    ]
    print("Nombre:       ", persona["nombre"])
    print("Instituciones:", ", ".join(instituciones))
    print("-" * 60)

Nombre:        Steve Wozniak
Instituciones: UC Berkeley
------------------------------------------------------------


In [43]:
#Celda 9 — Paso 12: actualizar el resumen de Steve Wozniak

# update_one() modifica el PRIMER documento que cumpla el filtro.
# Primer argumento → filtro: a quién modificamos.
# Segundo argumento → qué cambiamos. $set cambia SOLO los campos indicados
# y deja el resto del documento intacto (sin $set se reemplazaría el documento).
resultado = coleccion.update_one(
    {"nombre": "Steve Wozniak"},
    {"$set": {"resumen": "Silicon Valley | Tech Speaker & Legend"}}
)

# matched_count  → cuántos documentos cumplían el filtro
# modified_count → cuántos se han modificado realmente
print("Documentos encontrados:", resultado.matched_count)
print("Documentos modificados:", resultado.modified_count)

# Volvemos a consultar para comprobar el cambio
woz = coleccion.find_one({"nombre": "Steve Wozniak"}, {"nombre": 1, "resumen": 1, "_id": 0})
print("Resumen actualizado:", woz["resumen"])

Documentos encontrados: 1
Documentos modificados: 1
Resumen actualizado: Silicon Valley | Tech Speaker & Legend


In [44]:
#Celda 10 — Paso 13: eliminar el curriculum de "Linus Torvalds"

# delete_one() borra el PRIMER documento que cumpla el filtro.
# (delete_many() borraría TODOS los que lo cumplan.)
resultado = coleccion.delete_one({"nombre": "Linus Torvalds"})

# deleted_count → cuántos documentos se han borrado (debería ser 1)
print("Documentos eliminados:", resultado.deleted_count)

# Comprobamos que ya no existe: find_one devuelve None si no encuentra nada
print("¿Sigue existiendo Linus?", coleccion.find_one({"nombre": "Linus Torvalds"}) is not None)

Documentos eliminados: 1
¿Sigue existiendo Linus? False


In [45]:
#Celda 11 — Paso 14: contabilizar documentos

# count_documents({}) con filtro vacío cuenta todos los documentos.
# Después de borrar a Linus deberían quedar 2.
total = coleccion.count_documents({})
print("Documentos en Curriculums:", total)

# También podemos listar quiénes quedan
print("Personas:", [doc["nombre"] for doc in coleccion.find({}, {"nombre": 1, "_id": 0})])

Documentos en Curriculums: 2
Personas: ['Bill Gates', 'Steve Wozniak']


In [46]:
#Celda 12 — Paso 15: crear un índice para el año de inicio de formación y listar los índices

# Un índice es como el índice de un libro: permite a MongoDB encontrar
# documentos por ese campo sin tener que revisar la colección entera.
# Como "formacion" es una lista, MongoDB crea un índice MULTIKEY
# (una entrada por cada elemento de la lista).
# 1 = orden ascendente (-1 sería descendente).
nombre_indice = coleccion.create_index([("formacion.inicio", 1)])
print("Índice creado:", nombre_indice)

# Listar los índices activos de la colección.
# Siempre existe "_id_", que MongoDB crea automáticamente sobre el campo _id.
print("\nÍndices activos:")
for indice in coleccion.list_indexes():
    print(f"- {indice['name']}  →  campos: {dict(indice['key'])}")

Índice creado: formacion.inicio_1

Índices activos:
- _id_  →  campos: {'_id': 1}
- formacion.inicio_1  →  campos: {'formacion.inicio': 1}


In [ ]:
#Celda 13 — Paso 16 (operación adicional): índice de texto y búsqueda por palabras clave

# Un ÍNDICE DE TEXTO permite buscar palabras dentro de campos de texto libre,
# como haría un buscador. MongoDB divide el texto en palabras (tokenización),
# ignora las palabras vacías ("the", "of", "and"...) y reduce cada palabra a su
# raíz (stemming: "computing" → "comput"). Son técnicas básicas de PLN.
#
# Lo creamos sobre "resumen" y "titular". Los textos están en inglés, así que
# indicamos default_language="english" para aplicar sus reglas de stemming.
# Nota: una colección solo puede tener UN índice de texto.
nombre_indice = coleccion.create_index(
    [("resumen", "text"), ("titular", "text")],
    default_language="english"
)
print("Índice de texto creado:", nombre_indice)


def buscar(texto):
    """Busca currículums que contengan alguna de las palabras de 'texto'."""
    # $text + $search usa el índice de texto para buscar las palabras.
    # {"$meta": "textScore"} devuelve una PUNTUACIÓN de relevancia:
    # cuanto más alta, mejor encaja el documento con la búsqueda.
    resultados = coleccion.find(
        {"$text": {"$search": texto}},
        {"nombre": 1, "titular": 1, "_id": 0, "relevancia": {"$meta": "textScore"}}
    ).sort([("relevancia", {"$meta": "textScore"})])   # los más relevantes primero

    print(f'\nBúsqueda: "{texto}"')
    for doc in resultados:
        print(f"  - {doc['nombre']} ({doc['titular']})  →  relevancia: {doc['relevancia']:.2f}")


# Ejemplos de búsqueda
buscar("Microsoft")
buscar("foundation")          # encuentra "Foundation" aunque esté en mayúscula
buscar("computing legend")    # varias palabras: basta con que aparezca alguna